# Signal model: XGBoost on kinematic features

Input: `kinematic_features/output/kinematic_features.csv` (run `kinematic_features/run_pipeline.ipynb` first).

- 5-fold `StratifiedGroupKFold` on `subject_group`, so no person is in both train and test (`splits/folds.csv`).
- Task is a one-hot input. Missing channels stay NaN; `has_xy` flags them.
- Class imbalance handled with `scale_pos_weight`.
- Metrics at sample level and subject level (mean probability per subject).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

ROOT = Path.cwd().resolve()
while not (ROOT / "models").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "models"))
from common import SEED, OUT, load_folds, report

META = ["sample_id", "source_dataset", "task", "label", "subject_id", "subject_group"]

## 1. Features and subject folds

In [2]:
df = pd.read_csv(ROOT / "kinematic_features" / "output" / "kinematic_features.csv")
df = df.merge(load_folds()[["subject_group", "fold"]], on="subject_group")
X = pd.concat([df.drop(columns=META + ["fold"]),
               pd.get_dummies(df["task"], prefix="task", dtype=float)], axis=1)
y = df["label"].to_numpy()
print(X.shape)
df.groupby("fold").agg(recordings=("sample_id", "size"), subjects=("subject_group", "nunique"), pd_share=("label", "mean"))

(2385, 47)


,recordings,subjects,pd_share
fold,,,
0,437,52,0.540046
1,500,52,0.526000
2,481,52,0.525988
3,516,55,0.505814
4,451,49,0.465632


## 2. Cross-validated training (out-of-fold predictions)

In [3]:
def make_model(y_train):
    pos = y_train.mean()
    return XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8,
                         colsample_bytree=0.8, scale_pos_weight=(1 - pos) / pos,
                         eval_metric="logloss", random_state=SEED)

prob = np.zeros(len(df))
for k in sorted(df["fold"].unique()):
    tr, te = df["fold"] != k, df["fold"] == k
    clf = make_model(y[tr]).fit(X[tr], y[tr])
    prob[te] = clf.predict_proba(X[te])[:, 1]
oof = df[["sample_id", "subject_group", "source_dataset", "task", "label", "fold"]].assign(prob=prob)

## 3. Results

In [4]:
res = report("signal_xgb", oof)


== signal_xgb ==
     model   level    n  accuracy  balanced_accuracy  sensitivity  specificity    f1  roc_auc  roc_auc_fold_std  bacc_fold_std
signal_xgb  sample 2385     0.724              0.724        0.711        0.737 0.725    0.766               NaN            NaN
signal_xgb subject  260     0.835              0.832        0.851        0.812 0.854    0.895             0.032          0.046


## 4. Final model on all data and feature importance

In [4]:
clf = make_model(y).fit(X, y)
clf.save_model(OUT / "signal_xgb.json")
pd.Series(clf.feature_importances_, X.columns).sort_values(ascending=False).head(15).to_frame("importance")

,importance
acceleration_std,0.071201
n_strokes,0.060972
duration_s,0.049428
path_length,0.039278
acceleration_max,0.036635
nca,0.036338
jerk_std,0.036290
velocity_max,0.035667
ncv,0.035184
pressure_cv,0.033960
